# Debiased Image Baseline — ResNet-18
Addresses the dataset shortcut problem found in `08_image_baseline.ipynb`.

### What the baseline showed
Subgroup analysis revealed the model learned source-level shortcuts rather than genuine fraud cues:
- MIDV (100% genuine): roc_auc = NaN — model trivially correct, no fraud to detect
- FMIDV (100% forged): roc_auc = NaN — model trivially correct, no genuine to compare
- FantasyID (mixed): roc_auc = **0.644** — the only real test, barely above random

### Two fixes applied
1. **Stronger augmentation** — stronger ColorJitter, RandomGrayscale, GaussianBlur make
   colour and resolution less reliable as shortcuts
2. **Source-balanced sampling** — WeightedRandomSampler gives each source_dataset equal
   representation per batch so MIDV (4000 images) does not dominate training

### What to look for in results
- FantasyID roc_auc should increase above 0.644 if shortcuts are reduced
- Overall F1/ROC-AUC may drop slightly — that is expected and honest
- GradCAM should show more attention to document content vs backgrounds

## 0 · Imports & reproducibility

In [ ]:
import sys, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc as sk_auc

sys.path.insert(0, r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\src")

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

from debiased_utils import (
    load_image_splits, print_split_summary, build_dataloaders,
    build_val_transform, build_model, unfreeze_backbone, make_loss_fn,
    fit_model, run_test_evaluation, predict_probs,
    compute_metrics, tune_threshold, evaluate,
    subgroup_metrics, GradCAM,
)

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False

SEED   = 42
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}  |  PyTorch {torch.__version__}")

## 1 · Paths & config

In [ ]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR     = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR  = PROJECT_ROOT / "notebook" / "results" / "image_debiased"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Baseline results for comparison
BASELINE_DIR = PROJECT_ROOT / "notebook" / "results" / "image_baseline"

TRAIN_CSV = DATA_DIR / "image_train_group_test.csv"
VAL_CSV   = DATA_DIR / "image_val_group_test.csv"
TEST_CSV  = DATA_DIR / "image_test_group_test.csv"

BATCH_SIZE  = 32
NUM_WORKERS = 0

STAGE1_EPOCHS   = 5
STAGE1_LR       = 1e-3
STAGE1_PATIENCE = 5

STAGE2_EPOCHS   = 20
BACKBONE_LR     = 1e-5
HEAD_LR         = 1e-4
STAGE2_PATIENCE = 6

print("Results dir:", RESULTS_DIR)

## 2 · Load splits

In [ ]:
splits = load_image_splits(train_csv=TRAIN_CSV, val_csv=VAL_CSV, test_csv=TEST_CSV)
print_split_summary(splits.train_df, splits.val_df, splits.test_df)

## 3 · Source distribution
This plot shows why balancing is needed — MIDV has 2x more images than FantasyID
in training, so without balancing it dominates every batch.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colours = ["#4C72B0", "#DD8452", "#55A868"]

for ax, (name, df) in zip(axes, [("Train", splits.train_df), ("Test", splits.test_df)]):
    src_class = df.groupby(["source_dataset","image_class"]).size().unstack(fill_value=0)
    src_class.plot(kind="bar", ax=ax, color=["#4C72B0","#DD8452"],
                   edgecolor="white", width=0.6)
    ax.set_title(f"{name} — rows per source × class")
    ax.set_xlabel(""); ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=0)
    ax.spines[["top","right"]].set_visible(False)

fig.suptitle("Source dataset composition", fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "source_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 4 · Build dataloaders
`balance_sources=True` activates `WeightedRandomSampler`.
Each source_dataset gets equal weight per batch regardless of dataset size.

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(
    splits=splits, batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS, balance_sources=True,
)
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

## 5 · Stage 1 — frozen backbone (head warmup)

In [ ]:
if (RESULTS_DIR / "stage1_best.pt").exists():
    print("Stage 1 already trained — skipping.")
else:
    model   = build_model(pretrained=True, freeze_backbone=True, dropout=0.3).to(device)
    loss_fn = make_loss_fn(splits.train_df, device=device)
    optimizer_s1 = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=STAGE1_LR, weight_decay=1e-4,
    )
    model, history_s1 = fit_model(
        model=model, train_loader=train_loader, val_loader=val_loader,
        loss_fn=loss_fn, optimizer=optimizer_s1, device=device,
        epochs=STAGE1_EPOCHS, threshold=0.5,
        early_stopping_patience=STAGE1_PATIENCE,
        monitor_metric="val_roc_auc",
        model_save_path=RESULTS_DIR / "stage1_best.pt",
    )
    display(history_s1.round(4))

## 6 · Stage 2 — full fine-tuning (backbone unfrozen)

In [ ]:
if (RESULTS_DIR / "stage2_best.pt").exists():
    print("Stage 2 already trained — skipping.")
else:
    unfreeze_backbone(model)
    backbone_params = [p for n, p in model.named_parameters() if "fc" not in n]
    head_params     = list(model.fc.parameters())
    optimizer_s2 = torch.optim.Adam(
        [{"params": backbone_params, "lr": BACKBONE_LR},
         {"params": head_params,     "lr": HEAD_LR}],
        weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_s2, T_max=STAGE2_EPOCHS, eta_min=1e-7,
    )
    model, history_s2 = fit_model(
        model=model, train_loader=train_loader, val_loader=val_loader,
        loss_fn=loss_fn, optimizer=optimizer_s2, device=device,
        epochs=STAGE2_EPOCHS, threshold=0.5,
        early_stopping_patience=STAGE2_PATIENCE,
        monitor_metric="val_roc_auc",
        scheduler=scheduler,
        model_save_path=RESULTS_DIR / "stage2_best.pt",
    )
    history = pd.concat([history_s1, history_s2], ignore_index=True)
    history["epoch_global"] = range(1, len(history) + 1)
    history.to_csv(RESULTS_DIR / "training_history.csv", index=False)
    display(history_s2.round(4))

## 7 · Load saved model (run this instead of training after first run)

In [ ]:
model   = build_model(pretrained=False, freeze_backbone=False, dropout=0.3).to(device)
model.load_state_dict(torch.load(RESULTS_DIR / "stage2_best.pt", map_location=device))
model.eval()

loss_fn = make_loss_fn(splits.train_df, device=device)

history    = pd.read_csv(RESULTS_DIR / "training_history.csv")
history_s1 = history[history["epoch"] <= 5].copy()
history_s2 = history[history["epoch"] > 5].copy()

print("Model loaded.")

## 8 · Training curves

In [ ]:
if "epoch_global" not in history.columns:
    history["epoch_global"] = range(1, len(history) + 1)

n_s1 = len(history_s1)
ep   = history["epoch_global"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(ep, history["train_loss"], label="train", color="#4C72B0")
axes[0].plot(ep, history["val_loss"],   label="val",   color="#DD8452")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(ep, history["val_f1"],      label="F1",      color="#55A868")
axes[1].plot(ep, history["val_roc_auc"], label="ROC-AUC", color="#C44E52")
axes[1].set_ylim(0.75, 1.01)
axes[1].set_title("Validation metrics"); axes[1].set_xlabel("Epoch"); axes[1].legend()

axes[2].plot(ep, history["lr"], color="#8172B2")
axes[2].set_title("Learning rate"); axes[2].set_xlabel("Epoch")
axes[2].set_yscale("log")

for ax in axes:
    ax.axvline(x=n_s1+0.5, color="gray", linestyle="--", alpha=0.7)
    ax.spines[["top","right"]].set_visible(False)

fig.suptitle("Debiased ResNet-18 — training overview", fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 9 · Threshold tuning on validation set

In [ ]:
val_y_true, val_y_prob = predict_probs(model, val_loader, device)
best_threshold, sweep_df = tune_threshold(val_y_true, val_y_prob, metric="f1")
sweep_df.to_csv(RESULTS_DIR / "threshold_sweep.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(sweep_df["threshold"], sweep_df["precision"], label="Precision", color="#4C72B0")
axes[0].plot(sweep_df["threshold"], sweep_df["recall"],    label="Recall",    color="#DD8452")
axes[0].plot(sweep_df["threshold"], sweep_df["f1"],        label="F1",        color="#55A868", lw=2)
axes[0].axvline(x=best_threshold, color="red", linestyle="--", label=f"Best t={best_threshold:.2f}")
axes[0].set_xlabel("Threshold"); axes[0].set_title("Metrics vs threshold")
axes[0].legend(); axes[0].spines[["top","right"]].set_visible(False)

fpr, tpr, _ = roc_curve(val_y_true, val_y_prob)
axes[1].plot(fpr, tpr, color="#C44E52", lw=2, label=f"Val ROC-AUC = {sk_auc(fpr,tpr):.4f}")
axes[1].plot([0,1],[0,1],"k--",lw=1)
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
axes[1].set_title("ROC Curve (validation)")
axes[1].legend(); axes[1].spines[["top","right"]].set_visible(False)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "threshold_and_roc_val.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Selected threshold: {best_threshold:.2f}")

## 10 · Test evaluation

In [ ]:
test_metrics = run_test_evaluation(
    model=model, test_loader=test_loader,
    loss_fn=loss_fn, device=device, threshold=best_threshold,
)

y_true_test, y_prob_test = predict_probs(model, test_loader, device)
test_pred_df = splits.test_df.reset_index(drop=True).copy()
test_pred_df["y_true"] = y_true_test
test_pred_df["y_prob"] = y_prob_test
test_pred_df["y_pred"] = (test_pred_df["y_prob"] >= best_threshold).astype(int)
test_pred_df.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metric_keys = ["accuracy","precision","recall","f1","roc_auc"]
metric_vals = [test_metrics[k] for k in metric_keys]
bar_colours = ["#4C72B0","#55A868","#DD8452","#C44E52","#8172B2"]
bars = axes[0].bar(metric_keys, metric_vals, color=bar_colours, edgecolor="white")
for bar, val in zip(bars, metric_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=9)
axes[0].set_ylim(0,1.12); axes[0].set_title("Debiased test metrics")
axes[0].spines[["top","right"]].set_visible(False)

cm = np.array(test_metrics["confusion_matrix"])
axes[1].imshow(cm, cmap="Blues")
axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
axes[1].set_xticklabels(["Pred Genuine","Pred Fraud"])
axes[1].set_yticklabels(["True Genuine","True Fraud"])
axes[1].set_title(f"Confusion matrix  (t={best_threshold:.2f})")
for i in range(2):
    for j in range(2):
        axes[1].text(j,i,str(cm[i,j]),ha="center",va="center",fontsize=14,
                     color="white" if cm[i,j]>cm.max()/2 else "black")

fpr_t, tpr_t, _ = roc_curve(y_true_test, y_prob_test)
axes[2].plot(fpr_t, tpr_t, color="#C44E52", lw=2,
             label=f"Test ROC-AUC = {sk_auc(fpr_t,tpr_t):.4f}")
axes[2].plot([0,1],[0,1],"k--",lw=1)
axes[2].set_xlabel("FPR"); axes[2].set_ylabel("TPR")
axes[2].set_title("ROC Curve (test)"); axes[2].legend()
axes[2].spines[["top","right"]].set_visible(False)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "test_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

### Score distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(y_prob_test[y_true_test==0], bins=40, alpha=0.65, color="#4C72B0", label="Genuine", density=True)
ax.hist(y_prob_test[y_true_test==1], bins=40, alpha=0.65, color="#DD8452", label="Fraud",   density=True)
ax.axvline(x=best_threshold, color="red", linestyle="--", label=f"Threshold = {best_threshold:.2f}")
ax.set_xlabel("Predicted fraud probability"); ax.set_ylabel("Density")
ax.set_title("Score distribution — debiased test set")
ax.legend(); ax.spines[["top","right"]].set_visible(False)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 11 · Subgroup analysis
**This is the key comparison with the baseline.**

Baseline FantasyID roc_auc was 0.644 — barely above random.
If the debiasing worked, FantasyID roc_auc should be meaningfully higher.

In [ ]:
def plot_subgroup(sg_df, group_col, ax_f1, ax_rc):
    df = sg_df.sort_values("f1", ascending=True)
    get_c = lambda v: "#55A868" if v>=0.85 else "#DD8452" if v>=0.70 else "#C44E52"
    ax_f1.barh(df[group_col].astype(str), df["f1"],
               color=[get_c(v) for v in df["f1"]], edgecolor="white")
    ax_f1.axvline(x=0.85, color="gray", linestyle="--", alpha=0.5)
    ax_f1.set_xlabel("F1"); ax_f1.set_xlim(0,1.05)
    ax_f1.spines[["top","right"]].set_visible(False)
    ax_rc.barh(df[group_col].astype(str), df["recall"],
               color=[get_c(v) for v in df["recall"]], edgecolor="white")
    ax_rc.axvline(x=0.85, color="gray", linestyle="--", alpha=0.5)
    ax_rc.set_xlabel("Recall"); ax_rc.set_xlim(0,1.05)
    ax_rc.spines[["top","right"]].set_visible(False)

subgroup_results = {}
for col in ["source_dataset","source_type","doc_type"]:
    if col not in test_pred_df.columns:
        print(f"Column '{col}' not found — skipping."); continue
    sg = subgroup_metrics(test_pred_df, col, threshold=best_threshold)
    sg.to_csv(RESULTS_DIR / f"test_metrics_by_{col}.csv", index=False)
    subgroup_results[col] = sg
    fig, axes = plt.subplots(1, 2, figsize=(13, max(3, len(sg)*0.5+1)))
    plot_subgroup(sg, col, axes[0], axes[1])
    fig.suptitle(f"Debiased — subgroup metrics by {col}", fontsize=12)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"subgroup_{col}.png", dpi=150, bbox_inches="tight")
    plt.show()
    display(sg.round(3))

## 12 · Comparison with baseline
Side-by-side subgroup metrics for source_dataset.
The key number to watch is FantasyID roc_auc — baseline was 0.644.

In [ ]:
baseline_sg_path = BASELINE_DIR / "test_metrics_by_source_dataset.csv"

if not baseline_sg_path.exists():
    print("Baseline subgroup CSV not found — run 08_image_baseline first.")
else:
    baseline_sg  = pd.read_csv(baseline_sg_path)
    debiased_sg  = subgroup_results.get("source_dataset")

    if debiased_sg is not None:
        merged = baseline_sg[["source_dataset","f1","recall","roc_auc"]].merge(
            debiased_sg[["source_dataset","f1","recall","roc_auc"]],
            on="source_dataset", suffixes=("_baseline","_debiased")
        )
        print("\nBaseline vs Debiased — by source_dataset:")
        display(merged.round(3))

        # Bar chart comparison
        sources = merged["source_dataset"].tolist()
        x = np.arange(len(sources)); width = 0.35

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        for ax, metric in zip(axes, ["f1","recall"]):
            ax.bar(x-width/2, merged[f"{metric}_baseline"],  width,
                   label="Baseline",  color="#4C72B0", alpha=0.85)
            ax.bar(x+width/2, merged[f"{metric}_debiased"], width,
                   label="Debiased", color="#DD8452", alpha=0.85)
            ax.set_xticks(x); ax.set_xticklabels(sources)
            ax.set_ylim(0,1.1); ax.set_title(f"{metric.upper()} by source_dataset")
            ax.set_ylabel(metric.upper()); ax.legend()
            ax.spines[["top","right"]].set_visible(False)

        fig.suptitle("Baseline vs Debiased — subgroup comparison", fontsize=12)
        fig.tight_layout()
        fig.savefig(RESULTS_DIR / "baseline_vs_debiased_comparison.png",
                    dpi=150, bbox_inches="tight")
        plt.show()

        merged.to_csv(RESULTS_DIR / "baseline_vs_debiased_subgroup.csv", index=False)

### Overall metrics comparison

In [ ]:
baseline_seed_csv = BASELINE_DIR / "seed_results.csv"

if baseline_seed_csv.exists():
    baseline_row = pd.read_csv(baseline_seed_csv).iloc[0]

    cmp = pd.DataFrame([
        {"model": "Baseline",  "f1": baseline_row["test_f1"],
         "roc_auc": baseline_row["test_roc_auc"],
         "recall": baseline_row["test_recall"],
         "precision": baseline_row["test_precision"]},
        {"model": "Debiased",  "f1": test_metrics["f1"],
         "roc_auc": test_metrics["roc_auc"],
         "recall": test_metrics["recall"],
         "precision": test_metrics["precision"]},
    ])
    display(cmp.round(4))
    cmp.to_csv(RESULTS_DIR / "overall_comparison.csv", index=False)

    metrics_cmp = ["f1","roc_auc","recall","precision"]
    x = np.arange(len(metrics_cmp)); width = 0.35

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(x-width/2, cmp[cmp["model"]=="Baseline"][metrics_cmp].values[0],
           width, label="Baseline",  color="#4C72B0", alpha=0.85)
    ax.bar(x+width/2, cmp[cmp["model"]=="Debiased"][metrics_cmp].values[0],
           width, label="Debiased", color="#DD8452", alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(["F1","ROC-AUC","Recall","Precision"])
    ax.set_ylim(0.7, 1.05); ax.set_title("Overall: Baseline vs Debiased")
    ax.set_ylabel("Score"); ax.legend()
    ax.spines[["top","right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "overall_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

## 13 · GradCAM
Compare with baseline GradCAM — heatmaps should focus more on document content and less on borders/backgrounds.

In [ ]:
from PIL import Image as PILImage

cam           = GradCAM(model, target_layer=model.layer4[-1])
val_transform = build_val_transform()

correct   = test_pred_df[test_pred_df["y_true"]==test_pred_df["y_pred"]].sample(
                n=min(4,(test_pred_df["y_true"]==test_pred_df["y_pred"]).sum()),
                random_state=SEED)
incorrect = test_pred_df[test_pred_df["y_true"]!=test_pred_df["y_pred"]].sample(
                n=min(4,(test_pred_df["y_true"]!=test_pred_df["y_pred"]).sum()),
                random_state=SEED)
sample_df = pd.concat([correct, incorrect]).reset_index(drop=True)

n = len(sample_df)
fig, axes = plt.subplots(2, n, figsize=(n*2.8, 5.5))
if n == 1: axes = [[axes[0]], [axes[1]]]

for col, (_, row) in enumerate(sample_df.iterrows()):
    pil_img = PILImage.open(row["image_path"]).convert("RGB")
    tensor  = val_transform(pil_img).unsqueeze(0).to(device)
    heatmap = cam(tensor)
    overlay = GradCAM.overlay(pil_img, heatmap, alpha=0.45)
    true_lbl = "genuine" if int(row["y_true"])==0 else "fraud"
    pred_lbl = "genuine" if int(row["y_pred"])==0 else "fraud"
    ok = true_lbl == pred_lbl
    axes[0][col].imshow(pil_img); axes[0][col].set_title(f"GT: {true_lbl}", fontsize=8)
    axes[0][col].axis("off")
    axes[1][col].imshow(overlay)
    axes[1][col].set_title(f"Pred: {pred_lbl}", fontsize=8,
                            color="#2ca02c" if ok else "#d62728")
    axes[1][col].axis("off")

fig.suptitle("Debiased GradCAM\nOriginal (top) | Heatmap (bottom)  ·  Green=correct  Red=wrong",
             fontsize=10)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "gradcam_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 14 · Final summary

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.axis("off")
tbl = ax.table(
    cellText=[[f"{test_metrics[k]:.4f}"] for k in ["accuracy","precision","recall","f1","roc_auc"]],
    rowLabels=["Accuracy","Precision","Recall","F1","ROC-AUC"],
    colLabels=["Score"], cellLoc="center", rowLoc="center", loc="center",
)
tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1.2, 1.8)
ax.set_title(f"Debiased ResNet-18 — Test Results\nthreshold={best_threshold:.2f}  |  seed={SEED}",
             fontsize=11, pad=18)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "final_summary_table.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nAll results saved to:", RESULTS_DIR)